## Task 1 — Dataset Loading and Data Understanding

### Prediction objective

Supervised **regression** to predict `TARGET_deathRate`: mean per-capita (per 100,000) cancer mortalities by county (2010–2016).

### Key columns (from data dictionary)

| Column | Meaning |
|---|---|
| `TARGET_deathRate` | Dependent variable; mean per-capita cancer mortalities |
| `avgAnnCount` | Mean annual reported cancer diagnoses |
| `avgDeathsPerYear` | Mean annual cancer mortalities |
| `incidenceRate` | Mean per-capita cancer diagnoses |
| `medIncome` | Median income per county |
| `popEst2015` | County population (2015) |
| `povertyPercent` | Percent of populace in poverty |
| `AvgHouseholdSize` | Mean household size |
| `Geography` | County name (identifier; not a model feature) |
| `binnedInc` | Median income binned by decile |
| Education / coverage / race % columns | ACS-style demographic and insurance shares |

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Project root is one level above notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import config as cfg

In [2]:
# Load cancer registry dataset
df = pd.read_csv(cfg.RAW_CSV, encoding=cfg.CSV_ENCODING)

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

Shape: 3,047 rows × 34 columns


,avgAnnCount,avgDeathsPerYear,TARGET_deathRate,incidenceRate,medIncome,popEst2015,povertyPercent,studyPerCap,binnedInc,MedianAge,...,PctPrivateCoverageAlone,PctEmpPrivCoverage,PctPublicCoverage,PctPublicCoverageAlone,PctWhite,PctBlack,PctAsian,PctOtherRace,PctMarriedHouseholds,BirthRate
0,1397.0,469,164.9,489.8,61898,260131,11.2,499.748204,"(61494.5, 125635]",39.3,...,NaN,41.6,32.9,14.0,81.780529,2.594728,4.821857,1.843479,52.856076,6.118831
1,173.0,70,161.3,411.6,48127,43269,18.6,23.111234,"(48021.6, 51046.4]",33.0,...,53.8,43.6,31.1,15.3,89.228509,0.969102,2.246233,3.741352,45.372500,4.333096
2,102.0,50,174.7,349.7,49348,21026,14.6,47.560164,"(48021.6, 51046.4]",45.0,...,43.5,34.9,42.1,21.1,90.922190,0.739673,0.465898,2.747358,54.444868,3.729488
3,427.0,202,194.8,430.4,44243,75882,17.1,342.637253,"(42724.4, 45201]",42.8,...,40.3,35.0,45.3,25.0,91.744686,0.782626,1.161359,1.362643,51.021514,4.603841
4,57.0,26,144.4,350.1,49955,10321,12.5,0.000000,"(48021.6, 51046.4]",48.3,...,43.9,35.1,44.0,22.7,94.104024,0.270192,0.665830,0.492135,54.027460,6.796657


### Data validation

In [3]:
df.dtypes.to_frame("dtype")

,dtype
avgAnnCount,float64
avgDeathsPerYear,int64
TARGET_deathRate,float64
incidenceRate,float64
medIncome,int64
popEst2015,int64
povertyPercent,float64
studyPerCap,float64
binnedInc,object
MedianAge,float64


In [4]:
print("Missing values (nonzero only)")
missing = df.isna().sum()
missing_pct = (100 * missing / len(df)).round(2)
missing_tbl = (
    pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
    .query("missing_count > 0")
    .sort_values("missing_count", ascending=False)
)
display(missing_tbl if len(missing_tbl) else pd.DataFrame({"note": ["No missing values"]}))

Missing values (nonzero only)


,missing_count,missing_pct
PctSomeCol18_24,2285,74.99
PctPrivateCoverageAlone,609,19.99
PctEmployed16_Over,152,4.99


In [5]:
n_dupes = int(df.duplicated().sum())
print(f"\nDuplicate rows: {n_dupes}")


Duplicate rows: 0


In [6]:
# Geography uniqueness (county-level ID)
if "Geography" in df.columns:
    print(f"Unique Geography values: {df['Geography'].nunique():,} / {len(df):,} rows")

Unique Geography values: 3,047 / 3,047 rows


In [7]:
# Target and key numeric summaries
key_cols = [
    cfg.TARGET,
    cfg.COL_MEDIAN_INCOME,
    cfg.COL_POVERTY,
    cfg.COL_AVG_HH_SIZE,
    cfg.MEDIAN_AGE_COL,
]
display(df[key_cols].describe().T)

# Invalid MedianAge (ages above a plausible max)
invalid_age_mask = df[cfg.MEDIAN_AGE_COL] > cfg.MEDIAN_AGE_MAX_VALID
n_invalid_age = int(invalid_age_mask.sum())
print(
    f"\nInvalid {cfg.MEDIAN_AGE_COL} > {cfg.MEDIAN_AGE_MAX_VALID}: "
    f"{n_invalid_age} rows (max={df[cfg.MEDIAN_AGE_COL].max():.1f})"
)
if n_invalid_age:
    display(df.loc[invalid_age_mask, ["Geography", cfg.MEDIAN_AGE_COL]].head(10))

,count,mean,std,min,25%,50%,75%,max
TARGET_deathRate,3047.0,178.664063,27.751511,59.7000,161.20,178.1,195.20,362.80
medIncome,3047.0,47063.281917,12040.090836,22640.0000,38882.50,45207.0,52492.00,125635.00
povertyPercent,3047.0,16.878175,6.409087,3.2000,12.15,15.9,20.40,47.40
AvgHouseholdSize,3047.0,2.479662,0.429174,0.0221,2.37,2.5,2.63,3.97
MedianAge,3047.0,45.272333,45.304480,22.3000,37.70,41.0,44.00,624.00



Invalid MedianAge > 100: 30 rows (max=624.0)


,Geography,MedianAge
100,"Seward County, Nebraska",458.4
181,"Sandoval County, New Mexico",469.2
225,"Pittsylvania County, Virginia",546.0
318,"Iosco County, Michigan",624.0
425,"Person County, North Carolina",508.8
606,"Mineral County, Montana",619.2
637,"Cass County, Nebraska",498.0
843,"Tangipahoa Parish, Louisiana",412.8
991,"Greene County, Virginia",481.2
1199,"Harrison County, Mississippi",424.8


In [8]:
# Non-numeric / categorical columns
non_numeric = df.select_dtypes(exclude=[np.number]).columns.tolist()
print("Non-numeric columns:", non_numeric)
for col in non_numeric:
    print(f"  {col}: nunique={df[col].nunique()}, sample={df[col].dropna().iloc[0]!r}")

Non-numeric columns: ['binnedInc', 'Geography']
  binnedInc: nunique=10, sample='(61494.5, 125635]'
  Geography: nunique=3047, sample='Kitsap County, Washington'


### Assumptions and columns excluded from modeling

**Exclusions (locked in `config.EXCLUDE_COLUMNS`):**

- `Geography` — high-cardinality county identifier; not a generalizable feature
- `binnedInc` — redundant with continuous `medIncome`
- `PctSomeCol18_24` — ~75% missing; dropping avoids heavy imputation bias

**Other assumptions for later preprocessing (Task 2):**

- Treat `MedianAge > 100` as missing, then median-impute (fit on train only)
- Median-impute remaining missing numeric columns (`PctEmployed16_Over`, `PctPrivateCoverageAlone`) on train only
- Keep `avgDeathsPerYear`, `avgAnnCount`, and `incidenceRate` as features for predictive performance, but note that they are strongly related to mortality and may overstate deployable accuracy

## Task 2 — Preprocessing and Train/Test Split

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

df_model = df.copy()

# Make implausible ages before imputation null
n_invalid_age = int((df_model[cfg.MEDIAN_AGE_COL] > cfg.MEDIAN_AGE_MAX_VALID).sum())
df_model.loc[
    df_model[cfg.MEDIAN_AGE_COL] > cfg.MEDIAN_AGE_MAX_VALID, cfg.MEDIAN_AGE_COL
] = np.nan
print(f"Set {n_invalid_age} MedianAge values > {cfg.MEDIAN_AGE_MAX_VALID} to NaN")

FEATURES = [c for c in df_model.columns if c != cfg.TARGET and c not in cfg.EXCLUDE_COLUMNS]
assert all(pd.api.types.is_numeric_dtype(df_model[c]) for c in FEATURES), (
    "All modeling features must be numeric after exclusions"
)

X = df_model[FEATURES]
y = df_model[cfg.TARGET]

Set 30 MedianAge values > 100 to NaN


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=cfg.TEST_SIZE,
    random_state=cfg.SEED,
)

print(f"Split ratio: {1 - cfg.TEST_SIZE:.0%}, {cfg.TEST_SIZE:.0%}")
print(f"random_state={cfg.SEED}")
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train mean={y_train.mean():.2f}, y_test mean={y_test.mean():.2f}")

Split ratio: 80%, 20%
random_state=42
X_train: (2437, 30), X_test: (610, 30)
y_train mean=178.61, y_test mean=178.89


In [11]:
# Median impute
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, FEATURES),
    ],
    remainder="drop",
)

preprocessor.fit(X_train)
X_train_prep = preprocessor.transform(X_train)
X_test_prep = preprocessor.transform(X_test)

print(f"X_train_prep: {X_train_prep.shape}, X_test_prep: {X_test_prep.shape}")
print(f"Train NaNs after transform: {np.isnan(X_train_prep).sum()}")
print(f"Test NaNs after transform: {np.isnan(X_test_prep).sum()}")

# Quick leakage sanity check: imputer statistics come from train
imputer = preprocessor.named_transformers_["num"].named_steps["imputer"]
print(f"Imputer n_features_in_={imputer.n_features_in_} (should equal {len(FEATURES)})")

X_train_prep: (2437, 30), X_test_prep: (610, 30)
Train NaNs after transform: 0
Test NaNs after transform: 0
Imputer n_features_in_=30 (should equal 30)


### Task 2 summary

- Dropped `Geography`, `binnedInc`, and `PctSomeCol18_24` (30 total features)
- Cleaned up age (`MedianAge > 100` → NaN)
- Created an 80/20 train/test split with (`random_state=42`)
- Built and fit a preprocessing pipeline (`median impute` → `StandardScaler`) on training data, then transformed the test set.

## Task 3 — Baseline Model Development

**Model choice:** `RandomForestRegressor` inside a sklearn `Pipeline` that reuses the Task 2 preprocessor (`median impute` → `StandardScaler` → RF).


**Assumptions:** same exclusions and near-leakage notes as Tasks 1–2; the full pipeline is fit end-to-end so preprocessing stays coupled to the model for reproducible scoring later.

In [12]:
import joblib
from sklearn.ensemble import RandomForestRegressor

cfg.MODELS_DIR.mkdir(parents=True, exist_ok=True)

model = RandomForestRegressor(
    n_estimators=cfg.RF_N_ESTIMATORS,
    max_depth=cfg.RF_MAX_DEPTH,
    min_samples_leaf=cfg.RF_MIN_SAMPLES_LEAF,
    random_state=cfg.SEED,
    n_jobs=cfg.RF_N_JOBS,
)

# reproducible pipeline: preprocess + model
pipe = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ]
)

pipe.fit(X_train, y_train)

joblib.dump(pipe, cfg.MODEL_PATH)

print("RandomForestRegressor Hyperparameters:")
print(
    f"  n_estimators={cfg.RF_N_ESTIMATORS}, max_depth={cfg.RF_MAX_DEPTH}, "
    f"min_samples_leaf={cfg.RF_MIN_SAMPLES_LEAF}, random_state={cfg.SEED}"
)


print(f"\nModel Saved -> {cfg.MODEL_PATH}")

RandomForestRegressor Hyperparameters:
  n_estimators=200, max_depth=16, min_samples_leaf=2, random_state=42

Model Saved -> /Users/collinkim/Documents/UChicago ADS/Summer 2026/MLOps/Assignments/mlops_assign4/models/cancer_rf_pipeline.joblib


In [13]:
# Smoke-check reload
pipe_reloaded = joblib.load(cfg.MODEL_PATH)
pred_smoke = pipe_reloaded.predict(X_test)
print(f"reload successful")

reload successful


## Task 4 — Baseline Model Evaluation

In [14]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

cfg.FIGURES_DIR.mkdir(parents=True, exist_ok=True)

y_pred_test = pipe.predict(X_test)

rmse = float(root_mean_squared_error(y_test, y_pred_test))
mae = float(mean_absolute_error(y_test, y_pred_test))
r2 = float(r2_score(y_test, y_pred_test))

baseline_metrics = {
    "dataset": "original_test",
    "n_rows": int(len(y_test)),
    "rmse": rmse,
    "mae": mae,
    "r2": r2,
}
pd.DataFrame([baseline_metrics])

,dataset,n_rows,rmse,mae,r2
0,original_test,610,19.05466,14.000362,0.55627


In [15]:
residuals = y_test.to_numpy() - y_pred_test

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_test, y_pred_test, alpha=0.45, edgecolors="none", s=28)
lims = [
    min(float(y_test.min()), float(y_pred_test.min())),
    max(float(y_test.max()), float(y_pred_test.max())),
]
ax.plot(lims, lims, "r--", linewidth=1, label="ideal")
ax.set_xlabel("Actual TARGET_deathRate")
ax.set_ylabel("Predicted TARGET_deathRate")
ax.set_title("Predicted vs Actual")
ax.legend()
fig.tight_layout()

pred_vs_actual_path = cfg.FIGURES_DIR / "baseline_pred_vs_actual.png"
fig.savefig(pred_vs_actual_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {pred_vs_actual_path}")


Saved: /Users/collinkim/Documents/UChicago ADS/Summer 2026/MLOps/Assignments/mlops_assign4/reports/figures/baseline_pred_vs_actual.png


/var/folders/h6/qh4xdv4n6ld89l43j3kxvdd40000gn/T/ipykernel_70375/474877594.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_pred_test, residuals, alpha=0.45, edgecolors="none", s=28)
ax.axhline(0, color="r", linestyle="--", linewidth=1)
ax.set_xlabel("Predicted TARGET_deathRate")
ax.set_ylabel("Residual (actual - predicted)")
ax.set_title("Residuals vs Predicted")
fig.tight_layout()

residuals_path = cfg.FIGURES_DIR / "baseline_residuals.png"
fig.savefig(residuals_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {residuals_path}")


Saved: /Users/collinkim/Documents/UChicago ADS/Summer 2026/MLOps/Assignments/mlops_assign4/reports/figures/baseline_residuals.png


/var/folders/h6/qh4xdv4n6ld89l43j3kxvdd40000gn/T/ipykernel_70375/1256354173.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


#### **Interpretation and limitations**

**Performance** 
* RMSE ≈ 19.1, MAE ≈ 14.0, R-squared ≈ 0.56
* Predictions move with actual death rates, but errors of ~14–19 per 100k are still large relative to many county-to-county differences

**Plots**
* Points hug the diagonal overall and residuals are roughly centered at zero, with more scatter and some shrinkage at the high end—extreme counties are harder

**Limitations** 
* Mortality-adjacent inputs (`avgDeathsPerYear`, `avgAnnCount`, `incidenceRate`) can make offline scores look better than a real deployment would

## Task 5 — Evidently AI Monitoring Setup

* Reference dataset= train (features + actual + prediction)
* Current = original test. Reports: `DataDriftPreset`, `TargetDriftPreset`, `RegressionPreset`.

In [17]:
from evidently import ColumnMapping
from evidently.metric_preset import DataDriftPreset, RegressionPreset, TargetDriftPreset
from evidently.report import Report

cfg.EVIDENTLY_DIR.mkdir(parents=True, exist_ok=True)

y_pred_train = pipe.predict(X_train)

ref = X_train.copy()
ref[cfg.TARGET] = y_train.to_numpy()
ref["prediction"] = y_pred_train

cur = X_test.copy()
cur[cfg.TARGET] = y_test.to_numpy()
cur["prediction"] = y_pred_test

column_mapping = ColumnMapping(
    target=cfg.TARGET,
    prediction="prediction",
    numerical_features=FEATURES,
    task="regression",
)


In [18]:
baseline_drift_report = Report(metrics=[DataDriftPreset(), TargetDriftPreset()])
baseline_drift_report.run(
    reference_data=ref, current_data=cur, column_mapping=column_mapping
)
baseline_drift_path = cfg.EVIDENTLY_DIR / "baseline_data_target_drift.html"
baseline_drift_report.save_html(str(baseline_drift_path))

baseline_perf_report = Report(metrics=[RegressionPreset()])
baseline_perf_report.run(
    reference_data=ref, current_data=cur, column_mapping=column_mapping
)
baseline_perf_path = cfg.EVIDENTLY_DIR / "baseline_regression_performance.html"
baseline_perf_report.save_html(str(baseline_perf_path))

baseline_drift_path, baseline_perf_path


/Users/collinkim/Documents/UChicago ADS/Summer 2026/MLOps/Assignments/mlops_assign4/.venv/lib/python3.9/site-packages/sklearn/metrics/_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/Users/collinkim/Documents/UChicago ADS/Summer 2026/MLOps/Assignments/mlops_assign4/.venv/lib/python3.9/site-packages/sklearn/metrics/_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/Users/collinkim/Documents/UChicago ADS/Summer 2026/MLOps/Assignments/mlops_assign4/.venv/lib/python3.9/site-packages/sklearn/metrics/_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  w

(PosixPath('/Users/collinkim/Documents/UChicago ADS/Summer 2026/MLOps/Assignments/mlops_assign4/reports/evidently/baseline_data_target_drift.html'),
 PosixPath('/Users/collinkim/Documents/UChicago ADS/Summer 2026/MLOps/Assignments/mlops_assign4/reports/evidently/baseline_regression_performance.html'))

**Checks configured**
- `DataDriftPreset`
    * Feature distribution shift between train and original test (sanity check before scenario drift)
- `TargetDriftPreset`
    * Shift in actual target / prediction distributions
- `RegressionPreset`
    * regression error structure on current vs reference

## Task 6 — Modified Test Dataset Creation 

In [19]:
test_original = X_test.copy()
test_original[cfg.TARGET] = y_test.to_numpy()

test_A = test_original.copy()
test_A[cfg.COL_MEDIAN_INCOME] = test_A[cfg.COL_MEDIAN_INCOME] + cfg.SCENARIO_A_INCOME_DELTA

test_AB = test_A.copy()
test_AB[cfg.COL_POVERTY] = test_AB[cfg.COL_POVERTY] + cfg.SCENARIO_B_POVERTY_DELTA

test_ABC = test_AB.copy()
test_ABC[cfg.COL_AVG_HH_SIZE] = test_ABC[cfg.COL_AVG_HH_SIZE] + cfg.SCENARIO_C_HH_SIZE_DELTA

scenarios = {
    "original": test_original,
    "A": test_A,
    "AB": test_AB,
    "ABC": test_ABC,
}

In [20]:
paths = {
    "original": cfg.TEST_ORIGINAL_CSV,
    "A": cfg.TEST_SCENARIO_A_CSV,
    "AB": cfg.TEST_SCENARIO_AB_CSV,
    "ABC": cfg.TEST_SCENARIO_ABC_CSV,
}
for name, frame in scenarios.items():
    frame.to_csv(paths[name], index=False)

list(paths.values())


[PosixPath('/Users/collinkim/Documents/UChicago ADS/Summer 2026/MLOps/Assignments/mlops_assign4/data/test_original.csv'),
 PosixPath('/Users/collinkim/Documents/UChicago ADS/Summer 2026/MLOps/Assignments/mlops_assign4/data/test_scenario_A.csv'),
 PosixPath('/Users/collinkim/Documents/UChicago ADS/Summer 2026/MLOps/Assignments/mlops_assign4/data/test_scenario_AB.csv'),
 PosixPath('/Users/collinkim/Documents/UChicago ADS/Summer 2026/MLOps/Assignments/mlops_assign4/data/test_scenario_ABC.csv')]

In [21]:
saved = {
    "original": pd.read_csv(cfg.TEST_ORIGINAL_CSV),
    "A": pd.read_csv(cfg.TEST_SCENARIO_A_CSV),
    "AB": pd.read_csv(cfg.TEST_SCENARIO_AB_CSV),
    "ABC": pd.read_csv(cfg.TEST_SCENARIO_ABC_CSV),
}
baseline = saved["original"]

shift_cols = [cfg.COL_MEDIAN_INCOME, cfg.COL_POVERTY, cfg.COL_AVG_HH_SIZE]
expected_deltas = {
    "original": {cfg.COL_MEDIAN_INCOME: 0, cfg.COL_POVERTY: 0, cfg.COL_AVG_HH_SIZE: 0},
    "A": {cfg.COL_MEDIAN_INCOME: cfg.SCENARIO_A_INCOME_DELTA, cfg.COL_POVERTY: 0, cfg.COL_AVG_HH_SIZE: 0},
    "AB": {
        cfg.COL_MEDIAN_INCOME: cfg.SCENARIO_A_INCOME_DELTA,
        cfg.COL_POVERTY: cfg.SCENARIO_B_POVERTY_DELTA,
        cfg.COL_AVG_HH_SIZE: 0,
    },
    "ABC": {
        cfg.COL_MEDIAN_INCOME: cfg.SCENARIO_A_INCOME_DELTA,
        cfg.COL_POVERTY: cfg.SCENARIO_B_POVERTY_DELTA,
        cfg.COL_AVG_HH_SIZE: cfg.SCENARIO_C_HH_SIZE_DELTA,
    },
}

validation_rows = []
for name, frame in saved.items():
    for col in shift_cols:
        actual_delta = (frame[col] - baseline[col]).mean()
        expected = expected_deltas[name][col]
        validation_rows.append(
            {
                "scenario": name,
                "column": col,
                "expected_delta": expected,
                "actual_delta": actual_delta,
                "match": np.isclose(actual_delta, expected),
            }
        )
        assert np.allclose(frame[col] - baseline[col], expected), f"{name}: {col} row deltas incorrect"

validation = pd.DataFrame(validation_rows)
validation


,scenario,column,expected_delta,actual_delta,match
0,original,medIncome,0,0.0,True
1,original,povertyPercent,0,0.0,True
2,original,AvgHouseholdSize,0,0.0,True
3,A,medIncome,-40000,-40000.0,True
4,A,povertyPercent,0,0.0,True
5,A,AvgHouseholdSize,0,0.0,True
6,AB,medIncome,-40000,-40000.0,True
7,AB,povertyPercent,20,20.0,True
8,AB,AvgHouseholdSize,0,0.0,True
9,ABC,medIncome,-40000,-40000.0,True


## Task 7 — Scenario-Based Model Scoring

Score original + A / AB / ABC with the trained pipeline. Actual `TARGET_deathRate` is unchanged across scenarios.

In [22]:
rows = []
for name, frame in saved.items():
    X_sc = frame[FEATURES]
    y_sc = frame[cfg.TARGET]
    y_pred = pipe.predict(X_sc)
    rows.append(
        {
            "scenario": name,
            "n_rows": len(frame),
            "rmse": float(root_mean_squared_error(y_sc, y_pred)),
            "mae": float(mean_absolute_error(y_sc, y_pred)),
            "r2": float(r2_score(y_sc, y_pred)),
            "mean_prediction": float(y_pred.mean()),
        }
    )

metrics_summary = pd.DataFrame(rows)
metrics_summary


,scenario,n_rows,rmse,mae,r2,mean_prediction
0,original,610,19.054660,14.000362,0.556270,179.631125
1,A,610,20.975674,16.032462,0.462290,185.613752
2,AB,610,22.112911,17.202058,0.402403,188.009598
3,ABC,610,21.472855,16.531036,0.436497,185.711457


In [23]:
cfg.REPORTS_DIR.mkdir(parents=True, exist_ok=True)
metrics_summary.to_csv(cfg.METRICS_SUMMARY_CSV, index=False)
cfg.METRICS_SUMMARY_CSV


PosixPath('/Users/collinkim/Documents/UChicago ADS/Summer 2026/MLOps/Assignments/mlops_assign4/reports/metrics_summary.csv')

#### How controlled input changes affected predictions and metrics

Labels are unchanged across scenarios, so metric shifts reflect prediction drift only.

- **A** (`medIncome − 40k`): mean prediction **179.6 → 185.6**; RMSE **19.1 → 21.0**, R² **0.56 → 0.46**
- **AB** (+ `povertyPercent + 20`): mean **188.0**; worst fit — RMSE **22.1**, R² **0.40**
- **ABC** (+ `AvgHouseholdSize + 2`): mean **185.7**; RMSE **21.5**, R² **0.44** — better than AB, still below original

Income and poverty shifts both push predictions up, so AB hurts most. ABC partially reverses that drift. Mean prediction moves before accuracy fully breaks down.
